# 12b — Train Face Model (Utterance-Level Average Pooling)

Re-trains the same **ViT-B/16** and **MobileNetV3-Small** backbones with a
**utterance-level training objective**.

In notebook 12, cross-entropy is computed independently on each of the K saved
frames (frame-level loss, label = utterance emotion).  Here, all K frames for
an utterance are forwarded through the model together, their logits are
**average-pooled**, and cross-entropy is computed **once on the pooled
prediction**.  This aligns the training signal directly with the utterance-level
metric used during evaluation.

| | Frame-level (nb 12) | Utterance-level (this nb) |
|---|---|---|
| Training loss unit | one frame | one utterance (≤ K frames avg-pooled) |
| Loss formula | mean CE over all frames in batch | CE on avg-logits per utterance |
| Eval | avg probs across frames → argmax | same |
| Backbone / head | identical | identical |

**Outputs** (per model):
```
../experiments/face_unimodal_utterance/<version>/<slug>/
    model/best_model.pt   ← best checkpoint (dev macro-F1)
    model/arch.txt
    history.csv
    config.json
    train_metrics.json
    classification_report_test.json
```

**Depends on:** `03_preprocess_face.ipynb` (metadata + PNG crops).
Reuses the same `.npy` pixel cache built by notebook 12 — no extra disk work.

In [1]:
import os, sys

_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'src')) else os.path.abspath(os.path.join(_here, '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

In [2]:
import json
import time
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import matplotlib.pyplot as plt

from src.config import (
    DEVICE,
    FACE_MODELS_TO_RUN,
    FACE_TRAIN_BATCH_SIZE, FACE_TRAIN_EPOCHS, FACE_TRAIN_LR,
    FACE_WEIGHT_DECAY, FACE_WARMUP_RATIO, FACE_GRAD_CLIP_NORM,
    FACE_FREEZE_EPOCHS, FACE_BACKBONE_LR_MULT, FACE_LABEL_SMOOTHING,
    FACE_HEAD_DROPOUT, FACE_BALANCED_SAMPLER,
    FACE_USE_CACHE, FACE_CACHE_DIR,
    VERSION,
)
from src.data.face_dataset import (
    load_face_metadata, get_transforms, build_cache,
    _cache_dir_path,                 # reuse same .npy cache as nb 12
    LABEL2ID, ID2LABEL, NUM_CLASSES,
)
from src.models.face_model import (
    load_face_model, get_param_groups,
    freeze_backbone, unfreeze_backbone, count_params,
)
from src.training.utils import set_seed, safe_mkdir_run_dir, slugify, write_json

# Output root for utterance-level experiments (separate from frame-level nb 12)
EXP_ROOT_UTT = '../experiments/face_unimodal_utterance'

print('Device         :', DEVICE)
print('Models to train:', FACE_MODELS_TO_RUN)
print('Experiment root:', EXP_ROOT_UTT)

Device         : cuda
Models to train: ['vit_b_16', 'mobilenet_v3_small']
Experiment root: ../experiments/face_unimodal_utterance


## Configuration overview

In [3]:
from src.config import (
    FACE_K_FRAMES, FACE_FRAME_STRIDE, FACE_IMG_SIZE,
)
print(f'  K frames per utterance (max padding dim) : {FACE_K_FRAMES}')
print(f'  Image size                               : {FACE_IMG_SIZE}x{FACE_IMG_SIZE}')
print(f'  Epochs                                   : {FACE_TRAIN_EPOCHS}')
print(f'  Head LR / Backbone LR                    : {FACE_TRAIN_LR:.1e} / {FACE_TRAIN_LR*FACE_BACKBONE_LR_MULT:.1e}')
print(f'  Label smoothing                          : {FACE_LABEL_SMOOTHING}')
print(f'  Freeze epochs                            : {FACE_FREEZE_EPOCHS}')
print(f'  Balanced sampler                         : {FACE_BALANCED_SAMPLER}')
print(f'  NPY cache                                : {FACE_USE_CACHE}  →  {FACE_CACHE_DIR}')
print(f'  Version                                  : {VERSION}')

  K frames per utterance (max padding dim) : 5
  Image size                               : 224x224
  Epochs                                   : 20
  Head LR / Backbone LR                    : 1.0e-04 / 1.0e-05
  Label smoothing                          : 0.1
  Freeze epochs                            : 3
  Balanced sampler                         : True
  NPY cache                                : True  →  ../Data/processed_meld_faces/cache
  Version                                  : v1


## Load metadata

In [4]:
train_df = load_face_metadata('train', version=VERSION)
dev_df   = load_face_metadata('dev',   version=VERSION)
test_df  = load_face_metadata('test',  version=VERSION)

def _utt_count(df):
    return df.groupby(['dialogue_id', 'utterance_id']).ngroups

print(f'Train  frames: {len(train_df):>6}  |  utterances: {_utt_count(train_df):>5}')
print(f'Dev    frames: {len(dev_df):>6}  |  utterances: {_utt_count(dev_df):>5}')
print(f'Test   frames: {len(test_df):>6}  |  utterances: {_utt_count(test_df):>5}')
print(f'Classes: {list(LABEL2ID.keys())}')

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\src\data\face_dataset.py:112: DtypeWarning: Columns (0: total_frames, 1: frame_idx, 2: pred_prob) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(meta_path)


Train  frames:  47800  |  utterances:  9783
Dev    frames:   5324  |  utterances:  1091
Test   frames:  12117  |  utterances:  2481
Classes: ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']


## UtteranceFaceDataset

Each `__getitem__` returns **all K face crops** for a single utterance, stacked
into a `(max_frames, 3, H, W)` tensor.  Utterances with fewer than `max_frames`
frames are zero-padded; the actual count is returned as `n_frames` so the
training loop can mask the padding before averaging.

In [5]:
class UtteranceFaceDataset(Dataset):
    """
    Utterance-level dataset.  Each item contains ALL K face-crop frames
    for a single utterance, stacked and zero-padded to *max_frames*.

    Returns
    -------
    frames   : FloatTensor (max_frames, 3, H, W) — normalised crops,
               zero-padded beyond n_frames
    n_frames : int  — actual number of valid frames
    label_id : int  — emotion class index
    d_id     : int  — dialogue_id
    u_id     : int  — utterance_id
    """

    def __init__(
        self,
        df: pd.DataFrame,
        transform,
        max_frames: Optional[int] = None,
        use_cache: bool = True,
        cache_dir: Optional[str] = None,
    ):
        # Sort frames within each utterance by selection_rank so the order is
        # reproducible, then group.
        sorted_df = df.sort_values(
            ['dialogue_id', 'utterance_id', 'selection_rank'],
            na_position='last',
        )
        groups = sorted_df.groupby(['dialogue_id', 'utterance_id'], sort=False)

        self.utterances: List[Dict] = []
        for (d_id, u_id), grp in groups:
            self.utterances.append({
                'd_id':      int(d_id),
                'u_id':      int(u_id),
                'label_id':  int(grp['label_id'].iloc[0]),
                'img_paths': grp['image_path'].tolist(),
            })

        self.transform  = transform
        self.max_frames = (
            max_frames
            if max_frames is not None
            else max(len(u['img_paths']) for u in self.utterances)
        )
        # Cache dir must exist to be used
        self._cache_dir = (
            cache_dir
            if (use_cache and cache_dir and os.path.isdir(cache_dir))
            else None
        )

    def __len__(self) -> int:
        return len(self.utterances)

    def __getitem__(self, idx: int):
        utt = self.utterances[idx]
        frames = []

        for img_path in utt['img_paths']:
            if self._cache_dir is not None:
                stem  = os.path.splitext(os.path.basename(img_path))[0]
                npy_p = os.path.join(self._cache_dir, stem + '.npy')
                if os.path.exists(npy_p):
                    img = Image.fromarray(np.load(npy_p), mode='RGB')
                else:
                    img = Image.open(img_path).convert('RGB')
            else:
                img = Image.open(img_path).convert('RGB')

            frames.append(self.transform(img))

        n_frames = len(frames)

        # Zero-pad to max_frames so all utterances in a batch have equal depth
        if n_frames < self.max_frames:
            pad = torch.zeros_like(frames[0])
            frames += [pad] * (self.max_frames - n_frames)

        frames_tensor = torch.stack(frames)   # (max_frames, 3, H, W)
        return frames_tensor, n_frames, utt['label_id'], utt['d_id'], utt['u_id']


print('UtteranceFaceDataset defined.')

UtteranceFaceDataset defined.


## Core helpers: forward, loss, evaluate

`_forward_utterances` flattens the `(B, K, C, H, W)` batch to `(B*K, C, H, W)`,
runs a single forward pass, reshapes back, masks padding, and returns
`(B, n_classes)` average-pooled logits.

In [6]:
def _forward_utterances(
    model: nn.Module,
    frames: torch.Tensor,
    n_frames: torch.Tensor,
    device: str,
) -> torch.Tensor:
    """
    Forward a batch of utterances and return average-pooled logits.

    Parameters
    ----------
    frames   : (B, max_K, 3, H, W)
    n_frames : (B,) number of valid frames per utterance (int tensor)

    Returns
    -------
    avg_logits : (B, n_classes)
    """
    B, K_max, C, H, W = frames.shape
    frames_flat = frames.view(B * K_max, C, H, W).to(device)   # (B*K_max, C, H, W)

    logits_flat = model(frames_flat)                            # (B*K_max, n_cls)
    n_cls  = logits_flat.shape[-1]
    logits = logits_flat.view(B, K_max, n_cls)                 # (B, K_max, n_cls)

    # Build mask: True for valid frames, False for padding
    mask = (
        torch.arange(K_max, device=device)
             .unsqueeze(0)                                      # (1, K_max)
             .lt(n_frames.to(device).unsqueeze(1))             # (B, K_max)
             .unsqueeze(-1)                                     # (B, K_max, 1)
             .float()
    )

    # Sum valid logits, divide by number of valid frames
    n_valid    = n_frames.to(device).float().unsqueeze(1)      # (B, 1)
    avg_logits = (logits * mask).sum(dim=1) / n_valid          # (B, n_cls)
    return avg_logits


def _ce_utterance(
    avg_logits: torch.Tensor,
    labels: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    """Standard cross-entropy on utterance-level average-pooled logits."""
    return nn.functional.cross_entropy(
        avg_logits, labels, label_smoothing=label_smoothing,
    )


@torch.no_grad()
def _evaluate_utt(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    label_smoothing: float = 0.0,
) -> Dict:
    """
    Utterance-level evaluation — each DataLoader item is already one utterance.
    Returns accuracy, macro_f1, weighted_f1, loss.
    """
    model.eval()
    all_preds, all_gts = [], []
    total_loss, n_batches = 0.0, 0

    for frames, n_frames, labels, _d, _u in loader:
        with torch.amp.autocast(device, enabled=(device == 'cuda')):
            avg_logits = _forward_utterances(model, frames, n_frames, device)

        loss = _ce_utterance(avg_logits, labels.to(device), label_smoothing)
        total_loss += loss.item()
        n_batches  += 1

        preds = avg_logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_gts.extend(labels.numpy().tolist())

    preds_arr = np.array(all_preds)
    gts_arr   = np.array(all_gts)
    return {
        'loss':        total_loss / max(n_batches, 1),
        'accuracy':    float((preds_arr == gts_arr).mean()),
        'macro_f1':    float(f1_score(gts_arr, preds_arr, average='macro',    zero_division=0)),
        'weighted_f1': float(f1_score(gts_arr, preds_arr, average='weighted', zero_division=0)),
    }


print('_forward_utterances / _ce_utterance / _evaluate_utt defined.')

_forward_utterances / _ce_utterance / _evaluate_utt defined.


## Training function: `train_face_utterance`

Identical training recipe to notebook 12 (same backbone, head, optimiser,
scheduler, freeze warm-up, balanced sampler, label smoothing) — the only
difference is that the loss is computed on the utterance-level pooled logits
rather than on individual frames.

In [7]:
def train_face_utterance(
    model_name: str,
    train_df: pd.DataFrame,
    dev_df: pd.DataFrame,
    test_df: pd.DataFrame,
    # ── Hyperparameters (default = config.py values) ──────────────────────
    batch_size: int         = 16,              # utterance batch size
    epochs: int             = FACE_TRAIN_EPOCHS,
    lr: float               = FACE_TRAIN_LR,
    weight_decay: float     = FACE_WEIGHT_DECAY,
    warmup_ratio: float     = FACE_WARMUP_RATIO,
    grad_clip: float        = FACE_GRAD_CLIP_NORM,
    label_smoothing: float  = FACE_LABEL_SMOOTHING,
    backbone_lr_mult: float = FACE_BACKBONE_LR_MULT,
    freeze_epochs: int      = FACE_FREEZE_EPOCHS,
    head_dropout: float     = FACE_HEAD_DROPOUT,
    balanced_sampler: bool  = FACE_BALANCED_SAMPLER,
    use_cache: bool         = FACE_USE_CACHE,
    cache_dir: str          = FACE_CACHE_DIR,
    version: str            = VERSION,
    device: str             = DEVICE,
    seed: int               = 42,
    num_workers: int        = 4,
) -> str:
    """
    Fine-tunes *model_name* with utterance-level average-pooling loss.
    Returns the run directory path.
    """
    set_seed(seed)

    run_dir   = safe_mkdir_run_dir(
        os.path.join(EXP_ROOT_UTT, version, slugify(model_name))
    )
    model_dir = os.path.join(run_dir, 'model')
    os.makedirs(model_dir, exist_ok=True)

    print(f"\n{'='*62}")
    print(f'  Face model (utterance)    : {model_name}')
    print(f'  Run dir                   : {run_dir}')
    print(f'  Device                    : {device}')
    print(f'  Batch size (utterances)   : {batch_size}')
    print(f'  Freeze epochs             : {freeze_epochs}')
    print(f'  Backbone LR / Head LR     : {lr*backbone_lr_mult:.2e} / {lr:.2e}')
    print(f'  Label smoothing           : {label_smoothing}')
    print(f'  Balanced sampler          : {balanced_sampler}')
    print(f"{'='*62}")

    # ── Build .npy cache (same location as nb 12 — no duplication) ────────
    cache_dirs = {}
    for split, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
        if use_cache:
            cdir = _cache_dir_path(split, version, cache_dir)
            if not os.path.isdir(cdir):
                build_cache(df, split, version=version, cache_dir=cache_dir)
            cache_dirs[split] = cdir
        else:
            cache_dirs[split] = None

    # ── Determine max_frames from training metadata ────────────────────────
    # (Use the global max so padding dim is consistent across all splits.)
    global_max_frames = int(
        train_df.groupby(['dialogue_id', 'utterance_id']).size().max()
    )
    print(f'  Max frames / utterance (padding dim) : {global_max_frames}')

    # ── Build utterance-level datasets ─────────────────────────────────────
    eval_tf = get_transforms(train=False)
    train_ds = UtteranceFaceDataset(
        train_df, get_transforms(train=True),
        max_frames=global_max_frames,
        use_cache=use_cache, cache_dir=cache_dirs['train'],
    )
    dev_ds = UtteranceFaceDataset(
        dev_df, eval_tf,
        max_frames=global_max_frames,
        use_cache=use_cache, cache_dir=cache_dirs['dev'],
    )
    test_ds = UtteranceFaceDataset(
        test_df, eval_tf,
        max_frames=global_max_frames,
        use_cache=use_cache, cache_dir=cache_dirs['test'],
    )
    print(f'  Train utterances : {len(train_ds)}  |  Dev : {len(dev_ds)}  |  Test : {len(test_ds)}')

    # ── Utterance-level balanced sampler ───────────────────────────────────
    if balanced_sampler:
        utt_labels   = [u['label_id'] for u in train_ds.utterances]
        cls_counts   = np.bincount(utt_labels, minlength=NUM_CLASSES).astype(float)
        cls_w        = 1.0 / np.maximum(cls_counts, 1)
        samp_w       = cls_w[utt_labels]
        sampler      = WeightedRandomSampler(
            weights     = torch.from_numpy(samp_w).float(),
            num_samples = len(utt_labels),
            replacement = True,
        )
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, sampler=sampler,
            num_workers=num_workers, pin_memory=True, drop_last=True,
        )
    else:
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True,
            num_workers=num_workers, pin_memory=True, drop_last=True,
        )

    dev_loader  = DataLoader(dev_ds,  batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    # ── Model ──────────────────────────────────────────────────────────────
    model  = load_face_model(model_name, num_classes=NUM_CLASSES,
                              head_dropout=head_dropout, device=device)
    params = count_params(model)
    print(f'  Params : {params["trainable"]:,} trainable / {params["total"]:,} total')

    if freeze_epochs > 0:
        freeze_backbone(model, model_name)
        n_active = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Backbone frozen : {n_active:,} params active for first {freeze_epochs} epoch(s).')

    # ── Optimiser + scheduler ──────────────────────────────────────────────
    param_groups = get_param_groups(model, model_name, lr, backbone_lr_mult)
    optimizer    = AdamW(param_groups, weight_decay=weight_decay)
    total_steps  = len(train_loader) * epochs
    scheduler    = OneCycleLR(
        optimizer,
        max_lr          = [lr * backbone_lr_mult, lr],
        total_steps     = total_steps,
        pct_start       = warmup_ratio,
        anneal_strategy = 'cos',
    )
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

    # ── Training loop ──────────────────────────────────────────────────────
    history = []
    best_f1 = -1.0
    t_start = time.time()

    for epoch in range(1, epochs + 1):

        # Unfreeze backbone after warm-up
        if epoch == freeze_epochs + 1:
            unfreeze_backbone(model)
            n_active = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f'\n  [Epoch {epoch}] Backbone unfrozen — {n_active:,} params now active.')

        model.train()
        ep_loss, ep_n = 0.0, 0

        for frames, n_frames, labels, _d, _u in train_loader:
            labels = labels.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast(device, enabled=(device == 'cuda')):
                avg_logits = _forward_utterances(model, frames, n_frames, device)
                loss       = _ce_utterance(avg_logits, labels, label_smoothing)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            ep_loss += loss.item()
            ep_n    += 1

        train_loss  = ep_loss / max(ep_n, 1)
        dev_metrics = _evaluate_utt(model, dev_loader, device, label_smoothing)

        row = {
            'epoch':        epoch,
            'train_loss':   round(train_loss, 6),
            'dev_loss':     round(dev_metrics['loss'], 6),
            'dev_acc':      round(dev_metrics['accuracy'], 6),
            'dev_macro_f1': round(dev_metrics['macro_f1'], 6),
            'dev_wt_f1':    round(dev_metrics['weighted_f1'], 6),
            'frozen':       epoch <= freeze_epochs,
        }
        history.append(row)

        frozen_tag = ' [frozen]' if epoch <= freeze_epochs else ''
        print(
            f'  [Epoch {epoch:02d}/{epochs}]{frozen_tag} '
            f'train_loss={train_loss:.4f}  '
            f'dev_loss={dev_metrics["loss"]:.4f}  '
            f'dev_macro_f1={dev_metrics["macro_f1"]:.4f}'
        )

        if dev_metrics['macro_f1'] > best_f1:
            best_f1 = dev_metrics['macro_f1']
            torch.save(model.state_dict(), os.path.join(model_dir, 'best_model.pt'))
            print(f'    ↳ New best dev macro-F1: {best_f1:.4f} — checkpoint saved.')

    elapsed = time.time() - t_start

    # ── Save artefacts ─────────────────────────────────────────────────────
    pd.DataFrame(history).to_csv(os.path.join(run_dir, 'history.csv'), index=False)

    with open(os.path.join(model_dir, 'arch.txt'), 'w') as f:
        f.write(model_name)

    cfg = {
        'model_name':             model_name,
        'version':                version,
        'training_objective':     'utterance_level_avg_pooling',
        'epochs':                 epochs,
        'batch_size_utterances':  batch_size,
        'lr':                     lr,
        'backbone_lr_mult':       backbone_lr_mult,
        'freeze_epochs':          freeze_epochs,
        'weight_decay':           weight_decay,
        'warmup_ratio':           warmup_ratio,
        'grad_clip':              grad_clip,
        'label_smoothing':        label_smoothing,
        'head_dropout':           head_dropout,
        'balanced_sampler':       balanced_sampler,
        'num_classes':            NUM_CLASSES,
        'label2id':               LABEL2ID,
        'id2label':               {str(k): v for k, v in ID2LABEL.items()},
        'seed':                   seed,
        'device':                 device,
        'num_train_utterances':   len(train_ds),
        'num_dev_utterances':     len(dev_ds),
        'max_frames':             global_max_frames,
    }
    write_json(os.path.join(run_dir, 'config.json'), cfg)

    train_metrics = {
        'model_name':          model_name,
        'training_objective':  'utterance_level_avg_pooling',
        'best_dev_macro_f1':   round(best_f1, 6),
        'total_params':        params['total'],
        'trainable_params':    params['trainable'],
        'training_time_s':     round(elapsed, 1),
    }
    write_json(os.path.join(run_dir, 'train_metrics.json'), train_metrics)

    print(f'\n  Finished in {elapsed/60:.1f} min.  Best dev macro-F1: {best_f1:.4f}')
    print(f'  Artefacts → {run_dir}')
    return run_dir


print('train_face_utterance defined.')

train_face_utterance defined.


## Optional — hyperparameter overrides

`batch_size` here is the number of **utterances** per batch.  Each utterance
contains up to K frames, so the effective forward-pass size is
`batch_size × K_max`.  A value of 16 is recommended for ViT-B/16 on a 24 GB
GPU; increase to 32 for MobileNetV3 or if memory allows.

In [8]:
# ── Utterance batch size ───────────────────────────────────────────────────────
# 16 utterances × up to 5 frames = 80 images per forward pass (safe for ViT-B/16)
# Increase to 32 for MobileNetV3-Small or if GPU memory allows.
utt_batch_size = 16

epochs        = FACE_TRAIN_EPOCHS
lr            = FACE_TRAIN_LR
weight_decay  = FACE_WEIGHT_DECAY
warmup_ratio  = FACE_WARMUP_RATIO
grad_clip     = FACE_GRAD_CLIP_NORM

print(f'utt_batch_size={utt_batch_size}, epochs={epochs}, lr={lr}, wd={weight_decay}')

utt_batch_size=16, epochs=20, lr=0.0001, wd=0.01


## Train

In [ ]:
run_dirs = []

for model_name in FACE_MODELS_TO_RUN:
    run_dir = train_face_utterance(
        model_name    = model_name,
        train_df      = train_df,
        dev_df        = dev_df,
        test_df       = test_df,
        batch_size    = utt_batch_size,
        epochs        = epochs,
        lr            = lr,
        weight_decay  = weight_decay,
        warmup_ratio  = warmup_ratio,
        grad_clip     = grad_clip,
        version       = VERSION,
        device        = DEVICE,
    )
    run_dirs.append((run_dir, model_name))

print('\nAll run directories:')
for rd, mn in run_dirs:
    print(f'  [{mn}] {rd}')


  Face model (utterance)    : vit_b_16
  Run dir                   : ../experiments/face_unimodal_utterance\v1\vit_b_16
  Device                    : cuda
  Batch size (utterances)   : 16
  Freeze epochs             : 3
  Backbone LR / Head LR     : 1.00e-05 / 1.00e-04
  Label smoothing           : 0.1
  Balanced sampler          : True
  Max frames / utterance (padding dim) : 5
  Train utterances : 9783  |  Dev : 1091  |  Test : 2481
  Params : 85,804,039 trainable / 85,804,039 total
  Backbone frozen : 5,383 params active for first 3 epoch(s).


: 

## Training history

In [ ]:
fig, axes = plt.subplots(1, len(run_dirs), figsize=(8 * len(run_dirs), 4), squeeze=False)

for ax, (run_dir, model_name) in zip(axes[0], run_dirs):
    hist_path = os.path.join(run_dir, 'history.csv')
    if not os.path.exists(hist_path):
        ax.set_title(f'{model_name} — no history.csv found')
        continue
    hist = pd.read_csv(hist_path)
    ax.plot(hist['epoch'], hist['train_loss'],   label='train loss',   marker='o', markersize=4)
    ax.plot(hist['epoch'], hist['dev_loss'],     label='dev loss',     marker='s', markersize=4)
    ax.plot(hist['epoch'], hist['dev_macro_f1'], label='dev macro-F1', marker='^', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_title(model_name.replace('_', '-') + ' (utterance-level loss)')
    ax.legend()

plt.suptitle('Face model — utterance-level training history', fontsize=13)
plt.tight_layout()
plt.show()

## Test set evaluation

Reloads the best checkpoint for each model, runs utterance-level inference on
the test split, and saves a `classification_report_test.json` in the same
format as the frame-level experiments (keys match `ID2LABEL`, compatible with
`99_results_for_thesis.ipynb`).

In [ ]:
# Re-build test DataLoader (eval transforms, no shuffle)
def _make_test_loader(df, version=VERSION, use_cache=FACE_USE_CACHE,
                      cache_dir=FACE_CACHE_DIR, batch_size=16, num_workers=4):
    """Builds a test UtteranceFaceDataset from the global metadata."""
    global_max = int(
        train_df.groupby(['dialogue_id', 'utterance_id']).size().max()
    )
    cdir = None
    if use_cache:
        cdir = _cache_dir_path('test', version, cache_dir)
        if not os.path.isdir(cdir):
            build_cache(df, 'test', version=version, cache_dir=cache_dir)
    ds = UtteranceFaceDataset(
        df, get_transforms(train=False),
        max_frames=global_max, use_cache=use_cache, cache_dir=cdir,
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=num_workers, pin_memory=True)


test_loader_eval = _make_test_loader(test_df)
label_names      = [ID2LABEL[i] for i in range(NUM_CLASSES)]

for run_dir, model_name in run_dirs:
    ckpt_path = os.path.join(run_dir, 'model', 'best_model.pt')
    if not os.path.exists(ckpt_path):
        print(f'[{model_name}] Checkpoint not found at {ckpt_path} — skipping.')
        continue

    model_eval = load_face_model(model_name, num_classes=NUM_CLASSES,
                                  head_dropout=0.0, device=DEVICE)
    model_eval.load_state_dict(
        torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    )
    model_eval.eval()

    all_preds, all_gts = [], []
    with torch.no_grad():
        for frames, n_frames, labels, _d, _u in test_loader_eval:
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
                avg_logits = _forward_utterances(model_eval, frames, n_frames, DEVICE)
            preds = avg_logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_gts.extend(labels.numpy().tolist())

    acc  = float(np.mean(np.array(all_preds) == np.array(all_gts)))
    mf1  = float(f1_score(all_gts, all_preds, average='macro',    zero_division=0))
    wf1  = float(f1_score(all_gts, all_preds, average='weighted', zero_division=0))

    print(f'\n{model_name} — Test results (utterance-level):')
    print(f'  Accuracy   : {acc:.4f}')
    print(f'  Macro F1   : {mf1:.4f}')
    print(f'  Weighted F1: {wf1:.4f}')
    print(classification_report(all_gts, all_preds, target_names=label_names, zero_division=0))

    report = classification_report(
        all_gts, all_preds, target_names=label_names,
        output_dict=True, zero_division=0,
    )
    rpt_path = os.path.join(run_dir, 'classification_report_test.json')
    with open(rpt_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f'  Saved → {rpt_path}')

    del model_eval
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Compare utterance-level vs. frame-level results

Quick comparison of macro-F1 on the test set between the two training regimes.

In [ ]:
import glob

rows = []

# Frame-level results (notebook 12)
for model_name in FACE_MODELS_TO_RUN:
    slug = slugify(model_name)
    rpt_paths = glob.glob(
        f'../experiments/face_unimodal/{VERSION}/{slug}/classification_report_test.json'
    )
    for p in rpt_paths:
        with open(p) as f:
            rpt = json.load(f)
        rpt_lc = {k.lower(): v for k, v in rpt.items()}
        rows.append({
            'model':     model_name,
            'training':  'frame-level',
            'macro_f1':  round(rpt_lc.get('macro avg', {}).get('f1-score', float('nan')), 4),
            'accuracy':  round(rpt_lc.get('accuracy', float('nan')), 4),
        })

# Utterance-level results (this notebook)
for run_dir, model_name in run_dirs:
    p = os.path.join(run_dir, 'classification_report_test.json')
    if not os.path.exists(p):
        continue
    with open(p) as f:
        rpt = json.load(f)
    rpt_lc = {k.lower(): v for k, v in rpt.items()}
    rows.append({
        'model':     model_name,
        'training':  'utterance-level',
        'macro_f1':  round(rpt_lc.get('macro avg', {}).get('f1-score', float('nan')), 4),
        'accuracy':  round(rpt_lc.get('accuracy', float('nan')), 4),
    })

comparison_df = pd.DataFrame(rows).sort_values(['model', 'training'])
print(comparison_df.to_string(index=False))